<a href="https://colab.research.google.com/github/oxedanda/pml_Assignment_4/blob/main/PyTorch_script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# 1. Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Hyperparameters
num_epochs = 5
batch_size = 64
learning_rate = 0.001
num_classes = 10  # MNIST has 10 digits (0-9)

# 3. MNIST dataset transformation
# ResNet expects 3-channel input, so we convert grayscale MNIST to 3 channels
# Also resize to 224x224 as expected by most pre-trained CNNs like ResNet
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.Grayscale(num_output_channels=3), # Convert to 3 channels
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # ImageNet stats
])

# 4. Load MNIST data
train_dataset = datasets.MNIST(root='./data',
                               train=True,
                               transform=transform,
                               download=True)
test_dataset = datasets.MNIST(root='./data',
                              train=False,
                              transform=transform)

train_loader = DataLoader(dataset=train_dataset,
                          batch_size=batch_size,
                          shuffle=True)
test_loader = DataLoader(dataset=test_dataset,
                         batch_size=batch_size,
                         shuffle=False)

# 5. Load pre-trained model (e.g., ResNet18)
model = models.resnet18(pretrained=True)

# Freeze all parameters in the network
# for param in model.parameters():
#     param.requires_grad = False

# Modify the final fully connected layer for MNIST (10 classes)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, num_classes) # Replace the classification head

model = model.to(device)

# 6. Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 7. Training loop
print("\nStarting training...")
for epoch in range(num_epochs):
    model.train() # Set model to training mode
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

        if (i+1) % 100 == 0:
            print (f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], '
                   f'Loss: {loss.item():.4f}')

    train_accuracy = 100 * correct_train / total_train
    print(f'Epoch [{epoch+1}/{num_epochs}] finished. Average Loss: {running_loss/len(train_loader):.4f}, '
          f'Training Accuracy: {train_accuracy:.2f}%')

    # 8. Evaluation after each epoch
    model.eval() # Set model to evaluation mode
    with torch.no_grad():
        correct_test = 0
        total_test = 0
        test_loss = 0.0
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()

        avg_test_loss = test_loss / len(test_loader)
        accuracy = 100 * correct_test / total_test
        print(f'Test Loss: {avg_test_loss:.4f}, Test Accuracy: {accuracy:.2f}%')

print("\nTraining complete!")


Using device: cuda


100%|██████████| 9.91M/9.91M [00:00<00:00, 60.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.63MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.7MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.9MB/s]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 133MB/s]



Starting training...
Epoch [1/5], Step [100/938], Loss: 0.0851
Epoch [1/5], Step [200/938], Loss: 0.2374
Epoch [1/5], Step [300/938], Loss: 0.0064
Epoch [1/5], Step [400/938], Loss: 0.0415
Epoch [1/5], Step [500/938], Loss: 0.1029
Epoch [1/5], Step [600/938], Loss: 0.0319
Epoch [1/5], Step [700/938], Loss: 0.0251
Epoch [1/5], Step [800/938], Loss: 0.0578
Epoch [1/5], Step [900/938], Loss: 0.0468
Epoch [1/5] finished. Average Loss: 0.0690, Training Accuracy: 97.96%
Test Loss: 0.0468, Test Accuracy: 98.41%
Epoch [2/5], Step [100/938], Loss: 0.1036
Epoch [2/5], Step [200/938], Loss: 0.0044
Epoch [2/5], Step [300/938], Loss: 0.0164
Epoch [2/5], Step [400/938], Loss: 0.0209
Epoch [2/5], Step [500/938], Loss: 0.0187
Epoch [2/5], Step [600/938], Loss: 0.0708
Epoch [2/5], Step [700/938], Loss: 0.0026
Epoch [2/5], Step [800/938], Loss: 0.0082
Epoch [2/5], Step [900/938], Loss: 0.0113
Epoch [2/5] finished. Average Loss: 0.0343, Training Accuracy: 98.92%
Test Loss: 0.0262, Test Accuracy: 99.06%


In [2]:
torch.save(model.state_dict(), "mnist_resnet18.pth")
print("Model saved as mnist_resnet18.pth")

Model saved as mnist_resnet18.pth
